In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew
import umap
from ripser import ripser
from persim import plot_diagrams, wasserstein
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H4_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数）の6分割スケール
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

# 次元数が極端に小さくなるのを防ぐため N_SAMPLES = 200 に設定
N_SAMPLES = 200
CUTOFF_RATIO = 0.05 # 上位5%の次元を足切り（ストップワード除去の模倣）
K_NEIGHBORS = 15

results = []
print("=== H4計算実験（Top-Nカットオフの無効性）を開始します ===")

def calculate_hubness_skewness(X, k=15):
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute').fit(X)
    indices = nn.kneighbors(X, return_distance=False)[:, 1:]
    counts = np.bincount(indices.flatten(), minlength=X.shape[0])
    return skew(counts)

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    # 最小でも次元数Dが20以上になるように補正（カットオフ後に空間が成立するため）
    D_features = max(20, total_trials // N_SAMPLES)

    # カットオフする次元数
    cut_idx = max(1, int(D_features * CUTOFF_RATIO))
    D_remaining = D_features - cut_idx

    print(f"\n[Step {step}/6] 試行: {total_trials:,} (元次元:{D_features} -> カットオフ後:{D_remaining})")

    # --- A. 初期値依存性を排除したベースライン生成 ---
    np.random.seed(42 + step)
    X_base = np.random.uniform(0, 0.2, size=(N_SAMPLES, D_features))

    # --- B. べき乗則的過分散（Heavy-tailed Variance）の注入 ---
    # テキストデータのZipf則を模倣し、次元順に分散が減衰する乗数を生成
    ranks = np.arange(1, D_features + 1)
    base_skew = 5.0 + (step * 3.0)
    power_law_multipliers = 1.0 + base_skew / (ranks ** 0.5)

    X_skewed = X_base * power_law_multipliers

    # --- C. 【重要】Top-Nカットオフの実行（上位5%の除去） ---
    # 最も分散の大きい上位次元をバッサリ切り捨てる（実務での前処理を再現）
    X_base_cut = X_base[:, cut_idx:]
    X_skewed_cut = X_skewed[:, cut_idx:]

    # --- D. カットオフ後の距離行列とUMAP ---
    dist_base_cut = pairwise_distances(X_base_cut + 1e-10, metric='cosine')
    dist_skewed_cut = pairwise_distances(X_skewed_cut + 1e-10, metric='cosine')

    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_base_cut = reducer.fit_transform(dist_base_cut)
    umap_skewed_cut = reducer.fit_transform(dist_skewed_cut)

    # --- E. キャリブレーション1：残留ハブネス歪度の計算 ---
    hub_skew_base = calculate_hubness_skewness(X_base_cut, k=K_NEIGHBORS)
    hub_skew_skewed = calculate_hubness_skewness(X_skewed_cut, k=K_NEIGHBORS)

    # --- F. キャリブレーション2：カットオフ後の位相的崩壊度 (Wasserstein距離) ---
    ph_base = ripser(dist_base_cut, distance_matrix=True, maxdim=1)['dgms']
    ph_skewed = ripser(dist_skewed_cut, distance_matrix=True, maxdim=1)['dgms']

    dgm_base_h0 = ph_base[0] if len(ph_base[0]) > 0 else np.array([[0, 0]])
    dgm_skewed_h0 = ph_skewed[0] if len(ph_skewed[0]) > 0 else np.array([[0, 0]])
    w_dist_h0 = wasserstein(dgm_base_h0, dgm_skewed_h0, matching=False)

    # 結果の記録
    results.append({
        'Step': step,
        'Original_D': D_features,
        'Remaining_D': D_remaining,
        'Residual_Hubness_Base': hub_skew_base,
        'Residual_Hubness_Skewed': hub_skew_skewed,
        'Wasserstein_H0_PostCutoff': w_dist_h0
    })

    # --- G. グラフ出力 ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Step {step}: Post-Cutoff Analysis (Remaining D={D_remaining})', fontsize=16)

    # UMAP散布図 (カットオフ後でも歪むことを視覚化)
    axes[0,0].scatter(umap_base_cut[:,0], umap_base_cut[:,1], s=10, alpha=0.6)
    axes[0,0].set_title(f'UMAP Base (Cutoff applied)\nHubness Skew: {hub_skew_base:.2f}')
    axes[0,1].scatter(umap_skewed_cut[:,0], umap_skewed_cut[:,1], s=10, alpha=0.6, color='red')
    axes[0,1].set_title(f'UMAP Skewed (Cutoff applied)\nHubness Skew: {hub_skew_skewed:.2f}')

    # パーシステンス図 (位相崩壊が防げていないことを視覚化)
    plot_diagrams(ph_base, ax=axes[1,0])
    axes[1,0].set_title('PD Base (Post-Cutoff)')
    plot_diagrams(ph_skewed, ax=axes[1,1])
    axes[1,1].set_title(f'PD Skewed (Post-Cutoff)\nW-Dist H0: {w_dist_h0:.3f}')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 集計とまとめグラフ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.set_xlabel('Total Trials (log scale)')
ax1.set_xscale('log')
ax1.set_ylabel('Residual Hubness Skewness', color='red')
ax1.plot(trial_scales, df_results['Residual_Hubness_Base'], marker='o', linestyle='--', color='lightcoral', label='Base Residual Hubness')
ax1.plot(trial_scales, df_results['Residual_Hubness_Skewed'], marker='o', color='red', label='Skewed Residual Hubness')
ax1.tick_params(axis='y', labelcolor='red')

ax2 = ax1.twinx()
ax2.set_ylabel('Topological Collapse (Wasserstein H0)', color='blue')
ax2.plot(trial_scales, df_results['Wasserstein_H0_PostCutoff'], marker='s', color='blue', label='Post-Cutoff H0 Deviation')
ax2.tick_params(axis='y', labelcolor='blue')

fig.suptitle('H4: Ineffectiveness of Top-N Cutoff', fontsize=14)
plt.title('Topology remains collapsed and hubness persists even after removing top 5% dimensions', fontsize=10)
fig.tight_layout()
plt.grid(True, which="both", ls="--", alpha=0.5)

# 凡例の結合
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.savefig(f"{OUT_DIR}/summary_h4_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.1 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=3ddeb3ec624b12ad1d86308238ea4779b4fd3f6002f504010bff000dce7d01fe
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H4計算実験（Top-Nカットオフの無効性）を開始します ===

[Step 1/6] 試行: 1,000 (元次元:20 -> カットオフ後:19)

[Step 2/6] 試行: 10,000 (元次元:50 -> カットオフ後:48)

[Step 3/6] 試行: 100,000 (元次元:500 -> カットオフ後:475)

[Step 4/6] 試行: 1,000,000 (元次元:5000 -> カットオフ後:4750)

[Step 5/6] 試行: 5,000,000 (元次元:25000 -> カットオフ後:23750)

[Step 6/6] 試行: 10,000,000 (元次元:50000 -> カットオフ後:47500)

実験完了。全データを H4_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>